# filex — daily ops dashboard

Quick walkthrough of the filex `nodes` table, plus a few
summary statistics and a `matplotlib`-style chart placeholder.


## 1. Connect to SQLite

Production data lives in `/data/instance.sqlite`. We use
stdlib `sqlite3` for ad-hoc analysis.


In [1]:
import sqlite3
import json
print('sqlite3 module version:', sqlite3.version)


sqlite3 module version: 2.6.0


In [2]:
con = sqlite3.connect('/data/instance.sqlite')
cur = con.cursor()
cur.execute("""
    SELECT s.name, COUNT(*) FROM nodes n
    JOIN storages s ON s.id = n.storage_id
    WHERE n.deleted_at IS NULL AND n.type = 'file'
    GROUP BY s.name
""")
cur.fetchall()


[('main',1248), ('s3-test',912), ('sftp-archive',44)]

## 2. Top extensions

Which file types dominate the corpus?


In [3]:
import re
cur.execute("SELECT name FROM nodes WHERE type='file' AND deleted_at IS NULL")
from collections import Counter
c = Counter()
for (name,) in cur.fetchall():
    m = re.search(r'\\.(\\w+)$', name)
    if m: c[m.group(1).lower()] += 1
c.most_common(5)


[('jpg', 420), ('pdf', 187), ('docx', 92), ('mp4', 68), ('mp3', 52)]

## 3. Plot

matplotlib bar chart of the breakdown. (Static fixture — no
real image embedded so the file stays small.)


In [4]:
import matplotlib.pyplot as plt
labels, counts = zip(*c.most_common(5))
plt.bar(labels, counts)
plt.title('Top 5 extensions')


<matplotlib.axes._subplots.AxesSubplot at 0x7f...>